In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find GPT5 project root (paths.py). Run Jupyter with cwd GPT5 or GPT5/notebooks."
    )
import paths

In [1]:
import openai
import pandas as pd

# Load the CSV file
df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")

# Display the first few rows after removing duplicates
print(df)
# df.columns
# print(len(df))

            QA_ID                                            context  \
0        Merge Q1  A man in his 30s with AIDS presented with acut...   
1       Merge Q10  Coagulation-related tests indicated a low clot...   
2      Merge Q100  Anterior segment optical coherence tomography ...   
3     Merge Q1000  The findings of the remainder of the examinati...   
4     Merge Q1001  B, Incisional biopsy reveals a necrotic epithe...   
...           ...                                                ...   
1292   Merge Q990  C, Fluorescein angiography demonstrates hyperf...   
1293   Merge Q992  Stains were weak for CD4 and were negative for...   
1294   Merge Q994  Multiple temperature measurements, including a...   
1295   Merge Q995  He underwent colostomy after unsuccessful surg...   
1296   Merge Q996  A plain radiograph of the left femur showed a ...   

                                       question_options answer_df3  \
0     What Is Your Diagnosis?\n\nA: Herpes simplex v...          

In [2]:
import openai

def generate_direct_prediction(context, question):
    """
    Queries GPT-5 with a clinical vignette (context) and a multiple-choice question (with embedded options).
    Returns only the predicted answer in the format: '[Letter]: [Answer Text]' (e.g., 'B: Femoral artery murmur').
    """
    prompt = f"""
You are given some context and a multiple-choice question.

Select the most appropriate answer from the options provided.

{context}

{question}

Provide your response in the following format:\n<answer>Option [letter]</answer>"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="gpt-5",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip()

    except Exception:
        return "Error"

In [3]:
import pandas as pd
from tqdm import tqdm
import os

output_path = paths.PREDICTIONS / "gpt5_predictions_on_Llama70B_removed.csv"

# If continuing from a previous batch, load the existing file and get already-completed indices
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    completed_ids = set(df_existing.index)
    print(f"✅ Loaded existing file with {len(completed_ids)} completed rows.")
else:
    df_existing = pd.DataFrame()
    completed_ids = set()

# Collect new results in a list of dicts
new_rows = []

# Iterate with progress bar
for idx, row in tqdm(df.iterrows(), total=len(df)):
    if idx in completed_ids:
        continue  # skip already processed

    pred = generate_direct_prediction(row["70B_After_Removal"], row["question_options"])
    
    result_row = row.to_dict()
    result_row["gpt5_direct_prediction"] = pred
    new_rows.append(result_row)

    # Write out after each row to ensure persistence
    df_batch = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_existing, df_batch], ignore_index=True)
    df_combined.to_csv(output_path, index=False)


100%|██████████████████████████████████████████████████| 1297/1297 [6:51:21<00:00, 19.03s/it]


In [3]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
output_path = paths.PREDICTIONS / "gpt5_predictions_on_Llama70B_removed.csv"
df = pd.read_csv(output_path)

# Extract the predicted letter from the format <answer>Option A</answer>
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        match = re.search(r"Option\s+([A-J])", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

df["gpt_letter"] = df["gpt5_direct_prediction"].apply(extract_letter_from_xml)

# Clean and standardize the ground truth answer
df["answer_letter"] = df["answer_df3"].astype(str).str.strip().str.upper()

# Compare predictions to ground truth
df["gpt_letter_match"] = df.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Convert to binary for std calculation
df["gpt_letter_binary"] = df["gpt_letter_match"].map({"Correct": 1, "Incorrect": 0})

# Compute overall accuracy
correct_count = df["gpt_letter_binary"].sum()
total_count = df["gpt_letter_binary"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")

# Per-data source accuracy and std
for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = source_df["gpt_letter_binary"].sum()
    total = source_df["gpt_letter_binary"].notna().sum()
    acc = correct / total if total > 0 else 0
    std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")
    
    print(f"Data Source: {source}")
    print(f"  Correct Predictions: {correct}")
    print(f"  Total Predictions: {total}")
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Std Dev: {std:.4f}\n")


Letter-Based Correct Predictions: 982
Total Predictions Compared: 1297
Letter Match Accuracy: 75.71%
Data Source: jama
  Correct Predictions: 477
  Total Predictions: 582
  Accuracy: 81.96%
  Std Dev: 0.3849

Data Source: medbullets
  Correct Predictions: 156
  Total Predictions: 207
  Accuracy: 75.36%
  Std Dev: 0.4319

Data Source: medxpert
  Correct Predictions: 179
  Total Predictions: 315
  Accuracy: 56.83%
  Std Dev: 0.4961

Data Source: mmlu
  Correct Predictions: 170
  Total Predictions: 193
  Accuracy: 88.08%
  Std Dev: 0.3248



In [4]:
import numpy as np

# Collect accuracies per data source
accuracies = []

for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = (source_df["gpt_letter_match"] == "Correct").sum()
    total = source_df["gpt_letter_match"].notna().sum()
    acc = correct / total if total > 0 else 0
    accuracies.append(acc)

# Calculate standard deviation
accuracy_std = np.std(accuracies, ddof=1)  # use ddof=1 for sample std deviation
print(f"Standard Deviation of Accuracy Across Data Sources: {accuracy_std:.4f}")

Standard Deviation of Accuracy Across Data Sources: 0.1353


In [ ]:
# --- Physician Origin subset recalculation ---
from pathlib import Path
import pandas as pd
import re
import numpy as np

physician_csv = Path(r"/home/yuexing/NeuRIPS25/Physician_Labels/Mar2_2026_Data/933_Clinician_Student_Majority_Vote.csv")
physician_origins = set(
    pd.read_csv(physician_csv, usecols=['Origin'])['Origin'].astype(str).str.strip()
)

if 'df' in locals() and isinstance(df, pd.DataFrame):
    df_eval = df.copy()
elif 'output_path' in locals():
    df_eval = pd.read_csv(output_path)
else:
    raise RuntimeError('Could not find dataframe `df` or `output_path` in this notebook state.')

if 'Origin' not in df_eval.columns:
    raise KeyError('`Origin` column is missing from evaluation dataframe.')

df_eval = df_eval[df_eval['Origin'].astype(str).str.strip().isin(physician_origins)].copy()
print(f"Physician-Origin subset rows: {len(df_eval)}")

if len(df_eval) == 0:
    raise ValueError('No overlapping Origin IDs found with physician CSV.')

# Build gpt_letter when not already present
if 'gpt_letter' not in df_eval.columns:
    pred_col = None
    for c in ['gpt5_direct_prediction', 'gpt4o_direct_prediction', 'majority_vote', 'GPT5_on_72B_SR']:
        if c in df_eval.columns:
            pred_col = c
            break
    if pred_col is None:
        raise KeyError('No supported prediction column found to derive `gpt_letter`.')

    def extract_letter(x):
        if not isinstance(x, str):
            return None
        m = re.search(r'Option\s*\[?([A-J])\]?|^\s*([A-J])\s*$', str(x).strip(), flags=re.IGNORECASE)
        if m:
            return (m.group(1) or m.group(2)).upper()
        return None

    if pred_col == 'GPT5_on_72B_SR':
        df_eval['gpt_letter'] = df_eval[pred_col].astype(str).str.strip().str.upper()
    elif pred_col == 'majority_vote' and 'answer_corr' in df_eval.columns:
        # For this notebook style majority_vote is often already a letter
        df_eval['gpt_letter'] = df_eval[pred_col].astype(str).str.strip().str.upper()
    else:
        df_eval['gpt_letter'] = df_eval[pred_col].apply(extract_letter)

# Build answer_letter
if 'answer_letter' not in df_eval.columns:
    if 'answer_corr' in df_eval.columns:
        df_eval['answer_letter'] = df_eval['answer_corr'].astype(str).str.strip().str.upper()
    else:
        raise KeyError('No `answer_corr` column available to build `answer_letter`.')

# Match + binary columns
if 'gpt_letter_match' not in df_eval.columns:
    df_eval['gpt_letter_match'] = np.where(
        df_eval['gpt_letter'] == df_eval['answer_letter'],
        'Correct',
        'Incorrect'
    )

df_eval['gpt_letter_binary'] = (df_eval['gpt_letter_match'] == 'Correct').astype(int)

correct_count = int(df_eval['gpt_letter_binary'].sum())
total_count = int(df_eval['gpt_letter_binary'].notna().sum())
accuracy = correct_count / total_count if total_count > 0 else 0.0

print('\n=== Physician-Origin Recalculation ===')
print(f"Correct Predictions: {correct_count}")
print(f"Total Predictions: {total_count}")
print(f"Accuracy: {accuracy:.2%}")

source_col = None
for c in ['data_source_df3', 'data_source_corr', 'data_source_corr_trainee']:
    if c in df_eval.columns:
        source_col = c
        break

if source_col:
    print(f"\nPer-data-source stats ({source_col}):")
    for source in df_eval[source_col].dropna().unique():
        source_df = df_eval[df_eval[source_col] == source]
        c = int(source_df['gpt_letter_binary'].sum())
        t = int(source_df['gpt_letter_binary'].notna().sum())
        acc = c / t if t > 0 else 0.0
        std = source_df['gpt_letter_binary'].std(ddof=1) if t > 1 else float('nan')
        print(f"  {source}: Correct={c}, Total={t}, Accuracy={acc:.2%}, Std={std:.4f}")
